# Transformer From Scratch — PyTorch Lightning Edition

This notebook solves the same **sequence reversal** task as `transformer-from-scratch.ipynb`,
using the identical Transformer architecture — but delegates all the training boilerplate
to **PyTorch Lightning**.

**What's the same:** architecture, dataset, hyperparameters, evaluation helpers.  
**What's different:** no manual training loop, no `.to(device)`, no `zero_grad/backward/step`.

**Goal:** show how Lightning separates *what your model does* from *how training runs*.

---
## Cell 1 — Install & Imports

In [ ]:
# Uncomment if lightning is not installed:
# !pip install lightning

import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import lightning as L
import matplotlib.pyplot as plt

torch.manual_seed(42)
print(f'PyTorch  : {torch.__version__}')
print(f'Lightning: {L.__version__}')

---
## Cell 2 — Config / Hyperparameters

Identical to the original notebook — nothing changes here.

In [ ]:
VOCAB_SIZE = 20   # tokens 0..19  (0 = <SOS>)
SEQ_LEN    = 10
SOS_TOKEN  = 0
D_MODEL    = 32
N_HEADS    = 4
D_FF       = 64
N_LAYERS   = 2
BATCH_SIZE = 64
MAX_EPOCHS = 50
LR         = 1e-3

print('Hyperparameters:')
for k, v in dict(VOCAB_SIZE=VOCAB_SIZE, SEQ_LEN=SEQ_LEN, D_MODEL=D_MODEL,
                  N_HEADS=N_HEADS, D_FF=D_FF, N_LAYERS=N_LAYERS,
                  BATCH_SIZE=BATCH_SIZE, MAX_EPOCHS=MAX_EPOCHS, LR=LR).items():
    print(f'  {k:<12} = {v}')

---
## Cell 3 — Dataset

Identical `ReversalDataset` from the original. No changes needed.

In [ ]:
class ReversalDataset(Dataset):
    def __init__(self, num_samples):
        self.src = torch.randint(1, VOCAB_SIZE, (num_samples, SEQ_LEN))
        self.tgt = self.src.flip(dims=[1])

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return self.src[idx], self.tgt[idx]

# Quick sanity check
ds = ReversalDataset(5)
src, tgt = ds[0]
print(f'Input:  {src.tolist()}')
print(f'Target: {tgt.tolist()}')
assert src.tolist() == tgt.tolist()[::-1], 'reversal check failed'

---
## Cell 4 — Transformer Architecture (copied verbatim)

All components from the original notebook: `PositionalEncoding`, `MultiHeadAttention`,
`FeedForward`, `EncoderBlock`, `DecoderBlock`, `Transformer`.

**Nothing changes here** — Lightning wraps the model, it doesn't touch the architecture.

In [ ]:
# --- Positional Encoding ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


# --- Scaled Dot-Product Attention ---
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attention_weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(attention_weights, V)
    return output, attention_weights


# --- Multi-Head Attention ---
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        attn_output, _ = scaled_dot_product_attention(Q, K, V, mask)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_o(attn_output)


# --- Feed-Forward ---
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))


# --- Encoder Block ---
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, src_mask)))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x


# --- Decoder Block ---
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, tgt_mask=None, src_mask=None):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_output, enc_output, src_mask)))
        x = self.norm3(x + self.dropout(self.ff(x)))
        return x


# --- Full Transformer ---
class Transformer(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, num_heads=N_HEADS,
                 d_ff=D_FF, num_layers=N_LAYERS, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.src_embedding = nn.Embedding(vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model)
        self.encoder_layers = nn.ModuleList(
            [EncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.decoder_layers = nn.ModuleList(
            [DecoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.fc_out = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def make_causal_mask(self, seq_len, device):
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device)).unsqueeze(0).unsqueeze(0)
        return mask

    def forward(self, src, tgt):
        batch_size = tgt.size(0)
        sos = torch.zeros(batch_size, 1, dtype=torch.long, device=tgt.device)
        tgt_input = torch.cat([sos, tgt[:, :-1]], dim=1)
        tgt_mask = self.make_causal_mask(tgt_input.size(1), tgt_input.device)

        src_emb = self.dropout(self.pos_encoding(
            self.src_embedding(src) * math.sqrt(self.d_model)
        ))
        enc_output = src_emb
        for layer in self.encoder_layers:
            enc_output = layer(enc_output)

        tgt_emb = self.dropout(self.pos_encoding(
            self.tgt_embedding(tgt_input) * math.sqrt(self.d_model)
        ))
        dec_output = tgt_emb
        for layer in self.decoder_layers:
            dec_output = layer(dec_output, enc_output, tgt_mask)

        return self.fc_out(dec_output)  # (batch, seq_len, vocab_size)


# Verify architecture
_t = Transformer()
print(f'Transformer parameters: {sum(p.numel() for p in _t.parameters()):,}')
del _t

---
## Cell 5 — Lightning Module

This is the key new piece. A `LightningModule` wraps your model and defines:
- `training_step` — what happens for one batch (replaces the inner loop body)
- `validation_step` — same, but for validation data
- `configure_optimizers` — which optimizer(s) to use

Notice what's **missing** compared to the original's `train_model()` helper:
- No `.to(device)` — Lightning moves tensors automatically
- No `optimizer.zero_grad()` / `loss.backward()` / `optimizer.step()` — Lightning calls these
- No epoch loop — `Trainer` handles that

In [ ]:
class TransformerLightning(L.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = Transformer(
            vocab_size=VOCAB_SIZE, d_model=D_MODEL, num_heads=N_HEADS,
            d_ff=D_FF, num_layers=N_LAYERS
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def training_step(self, batch, batch_idx):
        src, tgt = batch
        logits = self.model(src, tgt)                          # (batch, seq_len, vocab_size)
        loss = self.loss_fn(logits.view(-1, VOCAB_SIZE), tgt.view(-1))
        self.log('train_loss', loss, prog_bar=True, on_epoch=True, on_step=False)
        return loss

    def validation_step(self, batch, batch_idx):
        src, tgt = batch
        logits = self.model(src, tgt)
        loss = self.loss_fn(logits.view(-1, VOCAB_SIZE), tgt.view(-1))
        self.log('val_loss', loss, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=LR)


# Quick shape test (CPU, no training)
_lm = TransformerLightning()
_src = torch.randint(1, VOCAB_SIZE, (4, SEQ_LEN))
_tgt = _src.flip(dims=[1])
_out = _lm.model(_src, _tgt)
print(f'Output shape: {_out.shape}  (expected: [4, {SEQ_LEN}, {VOCAB_SIZE}])')
del _lm, _src, _tgt, _out

---
## Cell 6 — DataModule

A `LightningDataModule` bundles dataset construction and DataLoader creation.
This is optional — you could pass DataLoaders directly to `trainer.fit()` —
but it keeps all data logic in one place.

In [ ]:
class ReversalDataModule(L.LightningDataModule):
    def setup(self, stage=None):
        self.train_ds = ReversalDataset(5000)
        self.val_ds   = ReversalDataset(500)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=BATCH_SIZE, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=BATCH_SIZE)


dm = ReversalDataModule()
dm.setup()
print(f'Train batches: {len(dm.train_dataloader())}')
print(f'Val   batches: {len(dm.val_dataloader())}')

---
## Cell 7 — Training (3 lines)

The entire training loop is now:
```python
model   = TransformerLightning()
trainer = L.Trainer(max_epochs=50, accelerator='auto')
trainer.fit(model, datamodule=dm)
```

`accelerator='auto'` picks GPU if available, CPU otherwise — no manual `.to(device)` needed.

In [ ]:
lit_model = TransformerLightning()

trainer = L.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='auto',
    enable_model_summary=True,
    log_every_n_steps=10,
)

trainer.fit(lit_model, datamodule=dm)

---
## Cell 8 — Evaluation & Predictions

The evaluation helpers from the original notebook work unchanged.
We just pass `lit_model.model` (the inner `Transformer`) and a CPU-compatible test loader.

In [ ]:
# Helpers — identical to the original notebook

def evaluate_model(model, loader):
    """Compute token-level and sequence-level accuracy."""
    device = next(model.parameters()).device
    model.eval()
    correct_tokens = 0
    total_tokens = 0
    correct_seqs = 0
    total_seqs = 0

    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            output = model(src, tgt)
            preds = output.argmax(dim=-1)

            correct_tokens += (preds == tgt).sum().item()
            total_tokens   += tgt.numel()
            correct_seqs   += (preds == tgt).all(dim=1).sum().item()
            total_seqs     += tgt.size(0)

    return correct_tokens / total_tokens, correct_seqs / total_seqs


def show_predictions(model, dataset, n=5):
    """Show n example predictions."""
    device = next(model.parameters()).device
    model.eval()
    with torch.no_grad():
        for i in range(n):
            src, tgt = dataset[i]
            output = model(src.unsqueeze(0).to(device), tgt.unsqueeze(0).to(device))
            pred = output.argmax(dim=-1).squeeze(0).cpu()
            match = 'OK' if pred.tolist() == tgt.tolist() else 'WRONG'
            print(f'  Input:  {src.tolist()}')
            print(f'  Target: {tgt.tolist()}')
            print(f'  Pred:   {pred.tolist()}  [{match}]')
            print()

In [ ]:
inner_model = lit_model.model  # plain Transformer — no Lightning wrapper

token_acc, seq_acc = evaluate_model(inner_model, dm.val_dataloader())
print(f'Token accuracy:    {token_acc:.1%}')
print(f'Sequence accuracy: {seq_acc:.1%}')
print()
show_predictions(inner_model, dm.val_ds)

In [ ]:
# Plot training + validation loss from Lightning's CSV logger
import csv, os

log_dir = trainer.logger.log_dir if trainer.logger else None

if log_dir and os.path.exists(os.path.join(log_dir, 'metrics.csv')):
    epochs, train_losses, val_losses = [], [], []
    with open(os.path.join(log_dir, 'metrics.csv')) as f:
        for row in csv.DictReader(f):
            if row.get('train_loss'):
                epochs.append(float(row['epoch']))
                train_losses.append(float(row['train_loss']))
            if row.get('val_loss'):
                val_losses.append(float(row['val_loss']))

    plt.figure(figsize=(8, 4))
    plt.plot(epochs, train_losses, label='train_loss', linewidth=2)
    if val_losses:
        plt.plot(val_losses, label='val_loss', linewidth=2, linestyle='--')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Transformer (Lightning) — Training Curve')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('CSV log not found — re-run training or check trainer.logger.log_dir')

---
## Cell 9 — What Lightning Did For Us

Here's a side-by-side of what changed between the original notebook and this one:

| Boilerplate in original | How Lightning handles it |
|---|---|
| `device = torch.device('cuda' if ... else 'cpu')` | `accelerator='auto'` in `Trainer` |
| `src, tgt = src.to(device), tgt.to(device)` | Lightning moves batch tensors automatically |
| `model.train()` / `model.eval()` | Called automatically before each step type |
| `optimizer.zero_grad()` | Called by `Trainer` before `training_step` |
| `loss.backward()` | Called by `Trainer` after `training_step` returns |
| `optimizer.step()` | Called by `Trainer` after `backward()` |
| `for epoch in range(epochs):` | `max_epochs=50` in `Trainer` |
| `for src, tgt in train_loader:` | `Trainer` iterates the DataLoader |
| Manual print every N epochs | `self.log(...)` → progress bar + CSV file |
| `time.time()` bookkeeping | Built into `Trainer` output |

### What Lightning does NOT remove
- The **model architecture** — that's always yours to write
- The **loss function logic** — you choose and apply it in `training_step`
- The **optimizer choice** — you return it from `configure_optimizers`
- The **data pipeline** — `DataModule` is just an organized wrapper, not magic

### When is Lightning worth using?

**Yes, use Lightning when:**
- You want multi-GPU / TPU training without rewriting your loop
- You want structured logging, checkpointing, or early stopping out of the box
- Your team maintains multiple experiments and consistent structure matters

**Stick with raw PyTorch when:**
- You're learning — reading a manual loop teaches more than a `Trainer`
- Your training loop has unusual control flow (e.g., GAN alternating steps)
- You want zero dependencies and maximum transparency